# Generate the YAML file config input to nextflow for the Bayesian model

This one is for the SV normalized copy numbers


1. Find all valid files
2. Find all vallid genes
3. Compile a dictionary of options
4. Save the dictionary

## Compute and plot the score

In [ ]:
import numpy as np
import seaborn as sns
import os
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob
from datetime import date
from compute_bayes_scores_utils import (
    gather_best_model_results,
    make_01,
)
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler


def setup_dirs(outDir):
    figuresDir = os.path.join(outDir, "figures")
    tablesDir = os.path.join(outDir, "tables")
    dataDir = os.path.join(outDir, "data")
    os.makedirs(dataDir, exist_ok=True)
    os.makedirs(figuresDir, exist_ok=True)
    os.makedirs(tablesDir, exist_ok=True)
    return figuresDir, tablesDir, dataDir


outDir = "."
figuresDir, tablesDir, dataDir = setup_dirs(outDir)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

res_save_path = os.path.join(tablesDir, "best_model_results.csv")

### Gather information from the fitted models

In [ ]:
NF_OUT_DIR = '.'
paths = glob.glob(
    f"{NF_OUT_DIR}/**/**/best_model.yaml",
    recursive=True,
)
res = gather_best_model_results(paths)

res.to_csv(res_save_path, index=False)

## Compute the score & Plot

In [ ]:
# The relative importance of sd vs mean in the final score
sd_coeff = .66
res = pd.read_csv(res_save_path)

# Normalize mu and sigma
sam = res['sigma_alpha_max'].values.copy()
sam = StandardScaler().fit_transform(sam.reshape(-1, 1)).flatten()

mam = res['mu_alpha_max'].values.copy()
mam = StandardScaler().fit_transform(mam.reshape(-1, 1)).flatten()

res['sam'] = sam
res['mam'] = mam

# Compute the ecDNA score
res['score'] = (res['max_alpha']**2) * (sd_coeff * res['sam'] + (1-sd_coeff) * res['mam'])

# Normalize between 0 and 1
res['score'] = make_01(res['score'])
res.sort_values(by="score", ascending=False, inplace=True)

res.to_csv(res_save_path, index=False)
